Script for lock in:

 parameter: r (resistance)

 - Set V (Read) (can calculate I=V/r)
 - Set frequency (Read)
 - Read R ($\sqrt(X^2+Y^2)$), $\theta$ (r = R / I)
 - Voltage: A_B (Read)
 - Other: time constant, input range, sensitivity

 Read lock in (R, $\theta$) ever x seconds 
 Save to text file (also need to read temperatures from MXC) as MXC Temperature, Resistance (Ohms), Theta

Read MXC:
LakeShore

In [18]:
import pyvisa
from pyvisa import VisaIOError
from datetime import datetime
import Lockin_class as LC
import pandas as pd
%matplotlib tk
import matplotlib.pyplot as plt

In [3]:
rm = pyvisa.ResourceManager()

for s in rm.list_resources():
    try:
        lockin = rm.open_resource(s)
        print(s, lockin.query('*IDN?').strip())
    except VisaIOError:
        continue

GPIB0::18::INSTR 00F00200
GPIB1::4::INSTR Stanford_Research_Systems,SR860,005576,V1.51


In [ ]:
rm = pyvisa.ResourceManager()
lockin = rm.open_resource('GPIB1::4::INSTR')  # where the lockin is

if 0:  # to use GPIB
    lakeshore = rm.open_resource('GPIB1::12::INSTR')  # where the LakeShore is
elif:  # to use Ethernet
    ip_address = "192.168.1.3"
    port = 7777

    visa_resource = f"TCPIP::{ip_address}::{port}::SOCKET"

    rm = pyvisa.ResourceManager()
    lakeshore = rm.open_resource(visa_resource)  # where the LakeShore is

    lakeshore.read_termination = '\n'
    lakeshore.write_termination = '\n'
    lakeshore.timeout = 5000  # milliseconds
else:
    

print("Connected to:", lockin.query("*IDN?").strip())
print("Connected to:", lakeshore.query("*IDN?").strip())

Connected to: Stanford_Research_Systems,SR860,005576,V1.51
Connected to: LSCI,MODEL372,LSA2EY9,1.5


In [ ]:
Calibration = LC.lockin(lockin, lakeshore, Rbias=100e3)
Calibration.initialize(v=100e-6, f=777, n=0)

In [9]:
Calibration.log_data(
    "C:\\Users\\MINER\\Documents\\B13 Cryolab\\Test.txt",
    sampling_spacing=1,
    init_sleep=0,
    hours=2
    # seconds=5
)

In [12]:
data = pd.read_csv("C:\\Users\\MINER\\Documents\\B13 Cryolab\\Test.txt", delimiter = '\t')

In [21]:
plt.scatter(data['Temperature (K)'], data['Resistance (Ohms)'] / 100, s=1)
plt.grid()
plt.xlabel("Temperature (K)")
plt.ylabel("Resistance (Ohms)")

Text(0, 0.5, 'Resistance (Ohms)')